In [1]:
import pandas as pd

df_marking = pd.read_csv('df_with_marking_final.csv')

Прошлые судимости здесь считаются все, не только 105 статья.

In [2]:
import spacy
from spacy.tokens import DocBin
import pandas as pd
import numpy as np

df_marking = pd.read_csv('/Users/ekaterinastepura/Downloads/df_with_marking_final (2).csv')
df_marking.loc[32, "prior_convictions"] = 'да'
df_marking.loc[45, "prior_convictions"] = 'да'
df_marking.loc[80, "prior_convictions"] = 'да'
df_marking.loc[81, "prior_convictions"] = 'да'
df_marking.loc[85, "prior_convictions"] = 'да'
df_marking.loc[92, "prior_convictions"] = 'да'
df_marking.loc[96, "prior_convictions"] = 'да'
df_marking.loc[97, "prior_convictions"] = 'да'
df_marking.loc[99, "prior_convictions"] = 'да'

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [3]:
print(df_marking["prior_convictions"].unique())
df_marking["prior_convictions"].value_counts()

['нет' "['нет', 'нет']" 'да' "['нет', 'да']"]


prior_convictions
нет               68
да                29
['нет', 'нет']     2
['нет', 'да']      1
Name: count, dtype: int64

In [4]:
import re
import pandas as pd

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

conviction_keywords = {
    'нет': ['не имеющего судимости', 'ранее не судимого', 'ранее не судим', 'не судимого', 'ранее не судимой', 'несудимого', 
            'не судимый', 'не судимой', 'ранее не судима', 'судимостей, влекущих правовые последствия, не имеющего', 'судимости не имеющего'],
    'да': ['судимого', 'ранее судимого', 'ранее судим', 'ранее был осужден', 'освобожден', 'ранее судимой', 'имеющего судимость', 'с учетом непогашенных судимостей', 'судимого:', 'судимой:', 'судимой']
}

train_data_prior_convictions = []

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    true_label = str(row.get("prior_convictions")).strip().lower()
    if true_label not in conviction_keywords:
        continue

    found = False
    for keyword in conviction_keywords[true_label]:
        match = re.search(r'\b' + re.escape(keyword) + r'\b', text_lower)
        if match:
            start, end = match.span()
            # print(f"✅ Найдено: '{text[start:end]}' (id={row['id']})")
            train_data_prior_convictions.append((text, {"entities": [(start, end, "PRIOR_CONVICTIONS")]}))
            found = True
            break

    if not found:
        train_data_prior_convictions.append((text, {"entities": []}))
        print(f"Не найдены ключевые слова '{true_label}' в id={idx}")
print(f"TRAIN_DATA_PRIOR_CONVICTIONS готово: {len(train_data_prior_convictions)} примеров")

Не найдены ключевые слова 'нет' в id=5
Не найдены ключевые слова 'нет' в id=7
Не найдены ключевые слова 'нет' в id=11
Не найдены ключевые слова 'нет' в id=22
Не найдены ключевые слова 'нет' в id=23
Не найдены ключевые слова 'нет' в id=24
Не найдены ключевые слова 'нет' в id=34
Не найдены ключевые слова 'нет' в id=47
Не найдены ключевые слова 'нет' в id=48
Не найдены ключевые слова 'нет' в id=61
Не найдены ключевые слова 'нет' в id=70
Не найдены ключевые слова 'нет' в id=74
Не найдены ключевые слова 'нет' в id=76
Не найдены ключевые слова 'нет' в id=79
Не найдены ключевые слова 'нет' в id=84
TRAIN_DATA_PRIOR_CONVICTIONS готово: 97 примеров


In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("PRIOR_CONVICTIONS")

examples = []
for text, annot in train_data_prior_convictions:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_prior_convictions_model")
print("Модель сохранена в 'ner_prior_convictions_model'")

Epoch 1, Losses: {'ner': 260316.2962059006}
Epoch 2, Losses: {'ner': 167.95925096933956}
Epoch 3, Losses: {'ner': 171.5152177887304}
Epoch 4, Losses: {'ner': 220.72627244437206}
Epoch 5, Losses: {'ner': 122.51386754482348}
Epoch 6, Losses: {'ner': 88.70933248763842}
Epoch 7, Losses: {'ner': 61.67416171077072}
Epoch 8, Losses: {'ner': 49.06128747406861}
Epoch 9, Losses: {'ner': 29.41747273503105}
Epoch 10, Losses: {'ner': 35.111021631200465}
Epoch 11, Losses: {'ner': 22.47601431822537}
Epoch 12, Losses: {'ner': 14.925720518519725}
Epoch 13, Losses: {'ner': 13.225950512568497}
Epoch 14, Losses: {'ner': 3.6059574745395855}
Epoch 15, Losses: {'ner': 19.164045111266237}
Модель сохранена в 'ner_prior_convictions_model'


In [5]:
import spacy
from sklearn.metrics import accuracy_score, f1_score

nlp_prior = spacy.load("ner_prior_convictions_model")

conviction_keywords = {
    'нет': ['не имеющего судимости', 'ранее не судимого', 'ранее не судим', 'не судимого', 'ранее не судимой', 'несудимого',
            'не судимый', 'не судимой', 'ранее не судима', 'судимостей, влекущих правовые последствия, не имеющего', 'судимости не имеющего'],
    'да': ['судимого', 'ранее судимого', 'ранее судим', 'ранее был осужден', 'освобожден', 'ранее судимой', 'имеющего судимость',
           'с учетом непогашенных судимостей', 'судимого:', 'судимой:', 'судимой']
}

flat_keyword_map = {}
for label, phrases in conviction_keywords.items():
    for phrase in phrases:
        flat_keyword_map[phrase.lower()] = label

y_true = []
y_pred = []

for idx, row in df_marking.iterrows():
    true_label = str(row.get("prior_convictions")).strip().lower()
    if true_label not in ["да", "нет"]:
        continue

    text = f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"
    doc = nlp_prior(text.lower())  

    predicted_label = "нет"  # по умолчанию — не судим

    for ent in doc.ents:
        if ent.label_ == "PRIOR_CONVICTIONS":
            ent_text = ent.text.strip().lower()
            for key_phrase, label in flat_keyword_map.items():
                if key_phrase in ent_text:
                    predicted_label = label
                    break
            break  # берём только первую сущность

    y_true.append(true_label)
    y_pred.append(predicted_label)

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, pos_label="да")

print(f"Accuracy: {acc:.2%}")
print(f"F1-score: {f1:.2%}")
print(f"Примеров: {len(y_true)}")

Accuracy: 96.91%
F1-score: 94.55%
Примеров: 97


Обучаем на 80, тестируем на 20.

In [ ]:
from sklearn.model_selection import train_test_split
from spacy.training import Example
import random

train_data, valid_data = train_test_split(train_data_prior_convictions, test_size=0.2, random_state=42)

train_examples = [Example.from_dict(nlp.make_doc(text), annot) for text, annot in train_data]
valid_examples = [Example.from_dict(nlp.make_doc(text), annot) for text, annot in valid_data]

In [ ]:
import spacy
from spacy.util import minibatch
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("PRIOR_CONVICTIONS")

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(train_examples)
    losses = {}
    batches = minibatch(train_examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Loss: {losses}")

Epoch 1, Loss: {'ner': 245199.20548655093}
Epoch 2, Loss: {'ner': 152.61718894741628}
Epoch 3, Loss: {'ner': 138.78024930998768}
Epoch 4, Loss: {'ner': 161.42471752467512}
Epoch 5, Loss: {'ner': 175.55240153202993}
Epoch 6, Loss: {'ner': 108.67389485567237}
Epoch 7, Loss: {'ner': 87.42587970251518}
Epoch 8, Loss: {'ner': 66.5963391184026}
Epoch 9, Loss: {'ner': 47.9630010550947}
Epoch 10, Loss: {'ner': 31.616377656556004}
Epoch 11, Loss: {'ner': 49.55179518625085}
Epoch 12, Loss: {'ner': 26.167603561727425}
Epoch 13, Loss: {'ner': 20.855739821432895}
Epoch 14, Loss: {'ner': 22.249817963050386}
Epoch 15, Loss: {'ner': 165.29208713252814}


In [ ]:
nlp.to_disk("ner_s_prior_convictions_model")
print("Модель сохранена в 'ner_s_prior_convictions_model'")